# Italian dataset

In [3]:
import os

folder_path = "ItalianDamagedRoad/data/labels-YOLO/"

for filename in os.listdir(folder_path):

    if filename.endswith(".txt"):

        file_path = os.path.join(folder_path, filename)

        with open(file_path, "r") as f:
            lines = f.readlines()

        # Keep only annotations whose class ID is NOT 2
        new_lines = [
            line for line in lines
            if line.strip() and line.split()[0] != "2"
        ]

        # Rewrite the file
        with open(file_path, "w") as f:
            f.writelines(new_lines)

# RDD2022

In [4]:
import os
import shutil

# RDD2022 label directories
pth_label_test = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/test/labels"
pth_label_train = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/train/labels"
pth_label_val = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/val/labels"

label_folders = [
    pth_label_train,
    pth_label_val,
    pth_label_test
]

# RDD2022 → unified classes
# 0 = pothole
# 1 = crack
mapping = {
    "0": "1",  # D00 longitudinal crack -> crack
    "1": "1",  # D10 transverse crack   -> crack
    "2": "1",  # D20 alligator crack    -> crack
    "3": "0",  # D40 pothole            -> pothole
}

for folder in label_folders:

    print(f"\nProcessing: {folder}")

    if not os.path.exists(folder):
        print(f"ERROR: Folder does not exist: {folder}")
        continue

    for filename in os.listdir(folder):

        if not filename.endswith(".txt"):
            continue

        file_path = os.path.join(folder, filename)

        new_lines = []

        with open(file_path, "r") as f:
            lines = f.readlines()

        for line_number, line in enumerate(lines, start=1):

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            # YOLO format:
            # class x_center y_center width height
            if len(parts) != 5:
                print(
                    f"WARNING: {filename}, line {line_number}: "
                    f"invalid YOLO format"
                )
                continue

            old_class = parts[0]

            if old_class not in mapping:
                print(
                    f"WARNING: {filename}, line {line_number}: "
                    f"unknown class {old_class}"
                )
                continue

            # Change class ID
            parts[0] = mapping[old_class]

            new_lines.append(" ".join(parts) + "\n")

        # Rewrite the label file
        with open(file_path, "w") as f:
            f.writelines(new_lines)

    print("Done.")

print("\nRDD2022 label unification completed.")


Processing: /home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/train/labels
Done.

Processing: /home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/val/labels
Done.

Processing: /home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/RDD2022/test/labels
Done.

RDD2022 label unification completed.


# SVRDD_YOLO

In [5]:
import os

main_folder = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/SVRDD_YOLO"

mapping = {
    "0": "1",
    "1": "1",
    "2": "1",
    "3": "0",
}

delete_classes = {"4", "5", "6"}

total_files = 0
modified_files = 0
deleted_annotations = 0
converted_annotations = 0

for root, dirs, files in os.walk(main_folder):

    for filename in files:

        if not filename.lower().endswith(".txt"):
            continue

        file_path = os.path.join(root, filename)
        total_files += 1

        with open(file_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        file_modified = False

        for line_number, line in enumerate(lines, start=1):

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) != 5:
                print(f"WARNING: Invalid YOLO label: {file_path} | line {line_number}")
                continue

            old_class = parts[0]

            if old_class in delete_classes:
                deleted_annotations += 1
                file_modified = True
                continue

            if old_class in mapping:
                new_class = mapping[old_class]

                if new_class != old_class:
                    file_modified = True

                parts[0] = new_class
                new_lines.append(" ".join(parts) + "\n")
                converted_annotations += 1

            else:
                print(f"WARNING: Unknown class '{old_class}' in {file_path} | line {line_number}")

        if file_modified:
            with open(file_path, "w") as f:
                f.writelines(new_lines)

            modified_files += 1

print(f"Total .txt files found: {total_files}")
print(f"Files modified: {modified_files}")
print(f"Annotations converted: {converted_annotations}")
print(f"Annotations deleted: {deleted_annotations}")

Total .txt files found: 8000
Files modified: 7338
Annotations converted: 10715
Annotations deleted: 10089


# UDTIRI

In [9]:
import os
import json

json_path = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/udtiri/annotations/instances_test2017.json"

json_dir = os.path.dirname(json_path)
labels_dir = os.path.join(json_dir, "labels-test")

os.makedirs(labels_dir, exist_ok=True)

with open(json_path, "r") as f:
    coco = json.load(f)

images = {
    image["id"]: image
    for image in coco["images"]
}

annotations_by_image = {}

for annotation in coco["annotations"]:
    image_id = annotation["image_id"]
    annotations_by_image.setdefault(image_id, []).append(annotation)

for image_id, image_info in images.items():

    file_name = image_info["file_name"]

    image_name = file_name.replace("\\", "/").split("/")[-1]
    image_number = os.path.splitext(image_name)[0]

    image_width = image_info["width"]
    image_height = image_info["height"]

    label_path = os.path.join(labels_dir, image_number + ".txt")

    yolo_lines = []

    for annotation in annotations_by_image.get(image_id, []):

        if annotation["category_id"] != 1:
            continue

        x, y, width, height = annotation["bbox"]

        x_center = (x + width / 2) / image_width
        y_center = (y + height / 2) / image_height
        width = width / image_width
        height = height / image_height

        yolo_lines.append(
            f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        )

    with open(label_path, "w") as f:
        f.write("\n".join(yolo_lines))

print(f"Conversion complete.")
print(f"Labels saved in: {labels_dir}")

KeyError: 'annotations'

# RDD2024

In [11]:
import os

folder = "/home/zinddine/Documents/VISION_YOLO/RoadAnomalyDetection/dataset/N-RDD2024Road damage and defects(1)"

mapping = {"0": "1", "1": "1", "2": "1", "4": "0"}
delete = {"3", "5", "6", "7", "8", "9"}

for root, _, files in os.walk(folder):
    for file in files:
        if not file.endswith(".txt"):
            continue

        path = os.path.join(root, file)

        with open(path) as f:
            lines = f.readlines()

        new_lines = []

        for line in lines:
            parts = line.split()

            if not parts:
                continue

            cls = parts[0]

            if cls in delete:
                continue

            if cls in mapping:
                parts[0] = mapping[cls]
                new_lines.append(" ".join(parts) + "\n")

        with open(path, "w") as f:
            f.writelines(new_lines)